# Trends diagnostic - Linear trend maps

The aim of this diagnostic is to compute and visualise the linear trend of a variable along time over a specified period and region. The trend is computed by polynomial fit through the AQUA `Trender` and rescaled to per-year units according to the inferred time frequency of the data.

In this notebook we demonstrate how to:
1. Compute a trend using the `Trends` class
2. Plot it using the `PlotTrends` class
3. Restrict the analysis to a region
4. Process multiple variables in a single run

In [ ]:
import os

import matplotlib

from aqua.diagnostics import PlotTrends, Trends

os.environ.pop("MPLBACKEND", None)
matplotlib.use("module://matplotlib_inline.backend_inline", force=True)

%reload_ext autoreload
%autoreload 2

## Setup data dictionaries

We define:
- `dataset_dict`: Configuration for the model data
- `common_dict`: Common parameters (analysis period, log level) reused by every example below

In [ ]:
dataset_dict = {"catalog": "climatedt-gen2", "model": "IFS-FESOM-5km", "exp": "baseline-hist", "source": "lra-r100-monthly"}

common_dict = {
    "startdate": "1990-01-01",
    "enddate": "1999-12-31",
    "loglevel": "INFO",
}

## Compute the trend

We create a `Trends` object on the global domain and call `run()` to:
1. Retrieve the variable from the catalog
2. Compute the linear trend coefficient along time and rescale it to per-year units
3. Save the result to a netCDF file

We analyse `2t` (2-metre air temperature).

In [ ]:
trend = Trends(**dataset_dict, **common_dict)
trend.run(var="2t")

## Plot the trend map

We pass the trend coefficients (`trend.trend_coef`) to `PlotTrends` and call `plot_trend()` to draw and save a 2D map.

In [ ]:
plot = PlotTrends(data=trend.trend_coef, loglevel="INFO")
plot.plot_trend(save_format="png", show=True)

## Regional trend

`Trends` accepts a `region` argument that is resolved against the centralized AQUA regions file (`aqua/diagnostics/config/definitions/regions.yaml`). The region name is propagated to the output filename and to the figure title.

Here we compute the same `2t` trend restricted to Europe.

In [ ]:
trend_europe = Trends(**dataset_dict, **common_dict, region="europe")
trend_europe.run(var="2t")

plot_europe = PlotTrends(data=trend_europe.trend_coef, loglevel="INFO")
plot_europe.plot_trend(save_format="png", show=True)

## Multi-variable trends

`Trends` also accepts a list of variables: the trend is computed for each in a single retrieval, and `PlotTrends` produces one figure per variable. Below we compute trends for both `2t` and `tprate`.

In [ ]:
trend_multi = Trends(**dataset_dict, **common_dict)
trend_multi.run(var=["2t", "tprate"])

plot_multi = PlotTrends(data=trend_multi.trend_coef, loglevel="INFO")
plot_multi.plot_trend(save_format="png", show=True)